In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# unzip the file
!unzip '/content/drive/MyDrive/Movie_Dataset/ml-32m.zip' -d '/content/drive/MyDrive/Movie_Dataset/unzipped_data'

Mounted at /content/drive
Archive:  /content/drive/MyDrive/Movie_Dataset/ml-32m.zip
   creating: /content/drive/MyDrive/Movie_Dataset/unzipped_data/ml-32m/
  inflating: /content/drive/MyDrive/Movie_Dataset/unzipped_data/ml-32m/tags.csv  
  inflating: /content/drive/MyDrive/Movie_Dataset/unzipped_data/ml-32m/links.csv  
  inflating: /content/drive/MyDrive/Movie_Dataset/unzipped_data/ml-32m/README.txt  
  inflating: /content/drive/MyDrive/Movie_Dataset/unzipped_data/ml-32m/checksums.txt  
  inflating: /content/drive/MyDrive/Movie_Dataset/unzipped_data/ml-32m/ratings.csv  
  inflating: /content/drive/MyDrive/Movie_Dataset/unzipped_data/ml-32m/movies.csv  


In [1]:
# remount google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

# explore the files in the unzipped folder

# path to the unzipped folder
folder_path = ('/content/drive/MyDrive/Movie_Dataset/unzipped_data/ml-32m')

# list all files in the unzipped folder
files = os.listdir(folder_path)

# print the list of files
print("files in the folder: ")
for file in files:
  print(file)

files in the folder: 
ratings.csv
tags.csv
movies.csv
links.csv
checksums.txt
README.txt


In [ ]:
import pandas as pd

# load ratings file
ratings = pd.read_csv('/content/drive/MyDrive/Movie_Dataset/unzipped_data/ml-32m/ratings.csv')

In [ ]:
# Inspect Ratings file
print("\nRatings Data:")
print(ratings.head())

print("\nRatings Info:")
print(ratings.info())

print("\nMissing Values in Ratings:")
print(ratings.isnull().sum())

print("\nDuplicates in Ratings:")
print(ratings.duplicated().sum())

print("\nNumber of Rows and Columns in Ratings:")
print(ratings.shape)

print("\nStatistics for Ratings:")
print(ratings['rating'].describe())


Ratings Data:
   userId  movieId  rating  timestamp
0       1       17     4.0  944249077
1       1       25     1.0  944250228
2       1       29     2.0  943230976
3       1       30     5.0  944249077
4       1       32     5.0  943228858

Ratings Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 976.6 MB
None

Missing Values in Ratings:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Duplicates in Ratings:
0

Number of Rows and Columns in Ratings:
(32000204, 4)

Statistics for Ratings:
count    3.200020e+07
mean     3.540396e+00
std      1.058986e+00
min      5.000000e-01
25%      3.000000e+00
50%      3.500000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64


In [ ]:
# drop timestamp column from ratings file
ratings = ratings.drop(columns=['timestamp'])

# ensuring that timestamp is dropped
print(ratings.head())

   userId  movieId  rating
0       1       17     4.0
1       1       25     1.0
2       1       29     2.0
3       1       30     5.0
4       1       32     5.0


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the ratings dataset
ratings = pd.read_csv("/content/drive/MyDrive/Movie_Dataset/unzipped_data/ml-32m/ratings.csv")

# Plot the distribution of ratings
plt.figure(figsize=(8,5))
plt.hist(ratings['rating'], bins=10, edgecolor='black', alpha=0.7)
plt.xlabel('Rating')
plt.ylabel('Count')
plt.title('Distribution of Ratings')
plt.xticks([0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5])
plt.show()

In [ ]:
# Check if userIds are contiguous
unique_users = ratings['userId'].unique()
is_users_contiguous = (unique_users.max() - unique_users.min() + 1) == len(unique_users)

# Check if movieIds are contiguous
unique_movies = ratings['movieId'].unique()
is_movies_contiguous = (unique_movies.max() - unique_movies.min() + 1) == len(unique_movies)

print("Are userIds contiguous?", is_users_contiguous)
print("Are movieIds contiguous?", is_movies_contiguous)

Are userIds contiguous? True
Are movieIds contiguous? False


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Remap movieIds to a contiguous range
movie_encoder = LabelEncoder()
ratings['movieId'] = movie_encoder.fit_transform(ratings['movieId'])

# Verify remapping
print("Unique movieIds after remapping:", ratings['movieId'].unique())

Unique userIds after remapping: [     1      2      3 ... 200946 200947 200948]
Unique movieIds after remapping: [   16    24    28 ... 46438 38505 78559]


In [ ]:
# Normalize rating column using (Mean-Centering)

# Compute the average rating for each user
user_mean = ratings.groupby('userId')['rating'].mean()

# Subtract the user's mean rating from each rating (Mean-Centering)
ratings['normalized_rating'] = ratings['rating'] - ratings['userId'].map(user_mean)

# Save the new dataset
ratings.to_csv('/content/drive/MyDrive/Movie_Dataset/unzipped_data/normalized.csv', index=False)

In [ ]:
import pandas as pd

# load normalized file
normalized = pd.read_csv('/content/drive/MyDrive/Movie_Dataset/unzipped_data/normalized.csv')

# Check if userIds are contiguous
unique_users = normalized['userId'].unique()
is_users_contiguous = (unique_users.max() - unique_users.min() + 1) == len(unique_users)

# Check if movieIds are contiguous
unique_movies = normalized['movieId'].unique()
is_movies_contiguous = (unique_movies.max() - unique_movies.min() + 1) == len(unique_movies)

print("Are userIds contiguous?", is_users_contiguous)
print("Are movieIds contiguous?", is_movies_contiguous)

In [ ]:
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix, save_npz
import pandas as pd

# Load normalized ratings
normalized = pd.read_csv('/content/drive/MyDrive/Movie_Dataset/unzipped_data/normalized.csv')

# Split the ratings into train/test (not users!)
train_df, test_df = train_test_split(normalized, test_size=0.2, random_state=42)

# Get shapes
num_users = normalized['userId'].max() + 1
num_movies = normalized['movieId'].max() + 1

# Create sparse matrices from splits
train_matrix = csr_matrix((train_df['normalized_rating'], (train_df['userId'], train_df['movieId'])),
                          shape=(num_users, num_movies))

test_matrix = csr_matrix((test_df['normalized_rating'], (test_df['userId'], test_df['movieId'])),
                         shape=(num_users, num_movies))

# Save them
save_npz('/content/drive/MyDrive/Movie_Dataset/unzipped_data/train_matrix.npz', train_matrix)
save_npz('/content/drive/MyDrive/Movie_Dataset/unzipped_data/test_matrix.npz', test_matrix)

In [ ]:
!pip install implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 71.1 MB/s eta 0:00:00


In [ ]:
from implicit.als import AlternatingLeastSquares
import numpy as np
from scipy.sparse import load_npz, save_npz
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from implicit.evaluation import precision_at_k
from implicit.evaluation import ndcg_at_k
from sklearn.metrics.pairwise import cosine_similarity
import pickle


# Load the sparse matrix
train_matrix = load_npz('/content/drive/MyDrive/Movie_Dataset/unzipped_data/train_matrix.npz')
test_matrix = load_npz('/content/drive/MyDrive/Movie_Dataset/unzipped_data/test_matrix.npz')


# Initialize the ALS model
model = AlternatingLeastSquares(
    factors=50,           # Number of latent factors
    iterations=20,        # Number of training iterations
    regularization=0.1,   # Regularization parameter
    random_state=42       # Random seed for reproducibility
)


# Train the model on the training set (sparse user-item matrix)
model.fit(train_matrix)

# Extract user and item indices from the testing set
test_users, test_items = test_matrix.nonzero()

# Predict ratings only for the testing set
predicted_ratings_test = np.array([model.user_factors[user] @ model.item_factors[item] for user, item in zip(test_users, test_items)])

# Extract actual ratings for the testing set (non-zero entries)
actual_ratings = test_matrix[test_users, test_items].A1  # .A1 converts to a flat array


# Evaluation metrics: RMSE, MAE, precision, recall, NDCG, coverage, diversity
rmse = np.sqrt(mean_squared_error(actual_ratings, predicted_ratings_test))
print("RMSE:", rmse)

mae = mean_absolute_error(actual_ratings, predicted_ratings_test)
print("MAE:", mae)

precision = precision_at_k(model, train_matrix, test_matrix, K=10)
print("Precision@10:", precision)

def recall_at_k(model, train_matrix, test_matrix, k=10):
    """
    Calculate Recall@k for the model without using model.recommend.

    Parameters:
    - model: Trained ALS model.
    - train_matrix: Sparse training matrix (used to avoid recommending already seen items).
    - test_matrix: Sparse testing matrix (contains the relevant items).
    - k: Number of recommendations to consider.

    Returns:
    - Average Recall@k across all users.
    """
    recall_scores = []

    # Get user and item factors from the model
    user_factors = model.user_factors
    item_factors = model.item_factors

    for user in range(test_matrix.shape[0]):
        # Get the actual relevant items for the user from the test set
        relevant_items = set(test_matrix[user].nonzero()[1])

        if len(relevant_items) == 0:
            continue  # Skip users with no relevant items in the test set

        # Get the predicted ratings for the user
        predicted_ratings = user_factors[user] @ item_factors.T

        # Mask out items already seen by the user in the training set
        seen_items = set(train_matrix[user].nonzero()[1])
        predicted_ratings[list(seen_items)] = -np.inf  # Set seen items to -inf to exclude them

        # Get the top-k recommended items
        top_k_items = np.argsort(predicted_ratings)[-k:][::-1]  # Sort and select top-k

        # Calculate Recall@k for the user
        recall = len(set(top_k_items).intersection(relevant_items)) / len(relevant_items)
        recall_scores.append(recall)

    # Return the average Recall@k across all users
    return np.mean(recall_scores)

# Calculate Recall@k
recall = recall_at_k(model, train_matrix, test_matrix, k=10)
print("Recall@10:", recall)


ndcg = ndcg_at_k(model, train_matrix, test_matrix, K=10)
print("NDCG@10:", ndcg)



def calculate_coverage(model, train_matrix, k=10, debug_users=5):
    """
    Calculate the coverage of the model's recommendations.

    Parameters:
    - model: Trained ALS model.
    - train_matrix: Sparse training matrix (used to avoid recommending already seen items).
    - k: Number of recommendations to consider.
    - debug_users: Number of users to print recommendations for (for debugging).

    Returns:
    - Coverage: Proportion of unique items recommended across all users.
    """
    recommended_items = set()

    for user in range(train_matrix.shape[0]):
        # Get the top-k recommendations for the user
        recommendations = model.recommend(user, train_matrix[user], N=k)

        # Debug: Print recommendations for the first few users
        if user < debug_users:
            print("Recommendations for user", user, ":", recommendations)

        # Handle the specific format returned by model.recommend
        if isinstance(recommendations, tuple) and len(recommendations) == 2:
            item_ids, scores = recommendations
            recommended_items.update(item_ids)
        else:
            raise ValueError("Recommendations are not in the expected format.")

    # Calculate coverage
    coverage = len(recommended_items) / train_matrix.shape[1]
    return coverage

# Calculate coverage
coverage = calculate_coverage(model, train_matrix, k=10)
print("Coverage:", coverage)


item_factors = model.item_factors
similarity_matrix = cosine_similarity(item_factors)
diversity = 1 - np.mean(similarity_matrix)
print("Diversity:", diversity)


# Save ALS model
with open('/content/drive/MyDrive/Movie_Dataset/als_model.pkl', 'wb') as f:
    pickle.dump(model, f)

/usr/local/lib/python3.11/dist-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

RMSE: 0.9083547947779461
MAE: 0.6866008006070728


  0%|          | 0/200749 [00:00<?, ?it/s]

Precision@10: 0.31490111737223464
Recall@10: 0.13440175567962837


  0%|          | 0/200749 [00:00<?, ?it/s]

NDCG@10: 0.3187174949756744
Recommendations for user 0 : (array([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], dtype=int32), array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32))
Recommendations for user 1 : (array([ 903,  840, 1183,  734, 1166, 2766, 1175, 1207, 1219, 1191],
      dtype=int32), array([0.58578795, 0.5775076 , 0.5048789 , 0.46712092, 0.45506117,
       0.43170726, 0.40487602, 0.3952735 , 0.3723217 , 0.36548895],
      dtype=float32))
Recommendations for user 2 : (array([359, 580, 579, 334, 148, 582,  33,  10,  16, 452], dtype=int32), array([0.33515444, 0.30086696, 0.2362282 , 0.22947985, 0.20699906,
       0.20236912, 0.1756444 , 0.17040722, 0.15926725, 0.14018938],
      dtype=float32))
Recommendations for user 3 : (array([ 582, 1258, 1939,  257,  344, 1190,   33,  579,  334, 1070],
      dtype=int32), array([0.56800157, 0.46492383, 0.41510725, 0.391885  , 0.3308689 ,
       0.32152048, 0.306158  , 0.26519212, 0.2613124 , 0.21950972],
      dtype=float32))
Recommendations f

In [ ]:
import pandas as pd

# load movie file
movies = pd.read_csv('/content/drive/MyDrive/Movie_Dataset/unzipped_data/ml-32m/movies.csv')

In [ ]:
# inspect the movies file

print("Movies Data:")
print(movies.head())

print("\nMovies Info:")
print(movies.info())

print("\nNumber of Rows and Columns in Movies:")
print(movies.shape)

print("\nMissing Values in Movies:")
print(movies.isnull().sum())

print("\nDuplicates in Movies:")
print(movies.duplicated().sum())

print("\nNumber of Unique Genres in Movies:")
print(movies['genres'].nunique())
print(movies['genres'].unique())

Movies Data:
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

Movies Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  87585 non-null  int64 
 1   title    87585 non-null  object
 2   genres   87585 non-null  object
dtypes: int64(1), object(2)
memory usage: 2.0+ MB
None

Number of Rows and Columns in

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import heapq
import pickle

# Load movie metadata and tags
movies = pd.read_csv('/content/drive/MyDrive/Movie_Dataset/unzipped_data/ml-32m/movies.csv')
tags = pd.read_csv('/content/drive/MyDrive/Movie_Dataset/unzipped_data/ml-32m/tags.csv')

# Step 1: Preprocess `movies`
movies['genres'] = movies['genres'].fillna('').apply(lambda x: x.replace('|', ' '))
movies['title'] = movies['title'].fillna('').apply(str)

# Step 2: Preprocess `tags.csv`
tags['tag'] = tags['tag'].fillna('').astype(str)

# Merge all tags per movie
tag_data = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
movies = pd.merge(movies, tag_data, on='movieId', how='left')
movies['tag'] = movies['tag'].fillna('')

# Step 3: Combine title, genres, and tags into a single text field
movies['cbf_text'] = (
    movies['title'] + ' ' +
    movies['genres'] + ' ' +
    movies['tag']
).str.lower()

# Step 4: TF-IDF vectorizer
tfidf = TfidfVectorizer(stop_words='english', min_df=5)
tfidf_matrix = tfidf.fit_transform(movies['cbf_text'])  # sparse matrix

# Step 5: Generate top-N similar movies for each movieId
def get_top_n_similar(tfidf_matrix, movies_df, n=10):
    top_similar = {}

    for idx in range(tfidf_matrix.shape[0]):
        sim_row = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
        top_indices = heapq.nlargest(n + 1, range(len(sim_row)), key=lambda x: sim_row[x])
        top_indices = [i for i in top_indices if i != idx][:n]

        base_movie_id = movies_df.iloc[idx]['movieId']
        similar_movies = [(movies_df.iloc[i]['movieId'], sim_row[i]) for i in top_indices]

        top_similar[base_movie_id] = similar_movies

    return top_similar

# Run similarity
top_n_sim = get_top_n_similar(tfidf_matrix, movies, n=10)

# Save to file
with open('/content/drive/MyDrive/Movie_Dataset/top_n_sim_cb.pkl', 'wb') as f:
    pickle.dump(top_n_sim, f)

print("✅ Enhanced CBF model saved with title + genres + tags!")


✅ Enhanced CBF model saved with title + genres + tags!


In [ ]:
import pandas as pd
import pickle
from collections import defaultdict
import numpy as np

# Load data
ratings = pd.read_csv('/content/drive/MyDrive/Movie_Dataset/unzipped_data/ml-32m/ratings.csv')

# Load top-N similarity dict (movieId as keys/values)
with open('/content/drive/MyDrive/Movie_Dataset/top_n_sim_cb.pkl', 'rb') as f:
    top_n_sim = pickle.load(f)

# Step 1: Build relevant and watched items per user
user_relevant = defaultdict(set)
user_watched = defaultdict(set)

for row in ratings.itertuples():
    if row.rating >= 4.0:
        user_relevant[row.userId].add(row.movieId)
    user_watched[row.userId].add(row.movieId)

# Step 2: Generate top-10 recommendations per user using CBF similarity
user_recs = defaultdict(list)

for user, relevant_movies in user_relevant.items():
    recs = set()
    for movie_id in relevant_movies:
        if movie_id in top_n_sim:
            sim_list = top_n_sim[movie_id]
            for sim_movie_id, score in sim_list:
                if sim_movie_id not in user_watched[user]:
                    recs.add(sim_movie_id)
                if len(recs) >= 10:
                    break
        if len(recs) >= 10:
            break
    user_recs[user] = list(recs)

# Step 3: Evaluate metrics
precisions, recalls, f1s, maps, ndcgs = [], [], [], [], []

for user in user_recs:
    rel = user_relevant[user]
    recs = user_recs[user]

    if not rel or not recs:
        continue

    hits = [1 if item in rel else 0 for item in recs]
    num_hits = sum(hits)

    # Precision@10
    precision = num_hits / 10
    # Recall@10
    recall = num_hits / len(rel)
    # F1@10
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    # MAP@10
    ap = 0.0
    correct = 0
    for i, hit in enumerate(hits, start=1):
        if hit:
            correct += 1
            ap += correct / i
    map_k = ap / min(len(rel), 10)

    # NDCG@10
    dcg = sum([hit / np.log2(idx + 2) for idx, hit in enumerate(hits)])
    ideal_hits = [1] * min(len(rel), 10)
    idcg = sum([1 / np.log2(idx + 2) for idx in range(len(ideal_hits))])
    ndcg = dcg / idcg if idcg != 0 else 0

    # Store results
    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)
    maps.append(map_k)
    ndcgs.append(ndcg)

# Final results
print("✅ Evaluation results (CBF):")
print(f"Precision@10: {np.mean(precisions):.4f}")
print(f"Recall@10:    {np.mean(recalls):.4f}")
print(f"F1@10:        {np.mean(f1s):.4f}")
print(f"MAP@10:       {np.mean(maps):.4f}")
print(f"NDCG@10:      {np.mean(ndcgs):.4f}")


✅ Evaluation results (CBF):
Precision@10: 0.0000
Recall@10:    0.0000
F1@10:        0.0000
MAP@10:       0.0000
NDCG@10:      0.0000


In [ ]:
import numpy as np
from collections import defaultdict
from sklearn.preprocessing import MinMaxScaler
import pickle
from scipy.sparse import load_npz

train_matrix = load_npz('/content/drive/MyDrive/Movie_Dataset/unzipped_data/train_matrix.npz')


with open('/content/drive/MyDrive/Movie_Dataset/als_model.pkl', 'rb') as f:
    model = pickle.load(f)

# Store ALS scores for each user
als_scores = defaultdict(dict)

num_users = train_matrix.shape[0]
num_items = train_matrix.shape[1]
top_k = 100  # Use top 100 to blend with CBF before filtering

print("Generating ALS recommendations...")

for user in range(num_users):
    recs = model.recommend(user, train_matrix[user], N=top_k, filter_already_liked_items=False)
    item_ids, scores = recs

    if len(scores) == 0:
        continue

    # Normalize ALS scores to [0, 1]
    scaler = MinMaxScaler()
    scores_norm = scaler.fit_transform(scores.reshape(-1, 1)).flatten()

    for item_id, score in zip(item_ids, scores_norm):
        als_scores[user][item_id] = score


# Save normalized ALS scores
with open('/content/drive/MyDrive/Movie_Dataset/als_scores.pkl', 'wb') as f:
    pickle.dump(als_scores, f)

Generating ALS recommendations...


In [ ]:
import pickle
from sklearn.preprocessing import MinMaxScaler
from collections import defaultdict
from scipy.sparse import load_npz
import numpy as np

# Load CBF similarity data
with open('/content/drive/MyDrive/Movie_Dataset/top_n_sim_cb.pkl', 'rb') as f:
    top_n_sim = pickle.load(f)

train_matrix = load_npz('/content/drive/MyDrive/Movie_Dataset/unzipped_data/train_matrix.npz')


# Step 2: Generate CBF scores per user
cbf_scores = defaultdict(dict)

print("Generating CBF similarity scores...")

num_users, num_items = train_matrix.shape

for user in range(num_users):
    liked_items = train_matrix[user].nonzero()[1]  # Items this user rated in training
    score_accumulator = defaultdict(float)

    for liked_item in liked_items:
        similar_items = top_n_sim.get(liked_item, [])
        for sim_item, sim_score in similar_items:
            if sim_item in liked_items:
                continue  # skip items already interacted with
            score_accumulator[sim_item] += sim_score  # accumulate similarity scores

    # Normalize CBF scores
    if score_accumulator:
        items, scores = zip(*score_accumulator.items())
        scaler = MinMaxScaler()
        scores_norm = scaler.fit_transform(np.array(scores).reshape(-1, 1)).flatten()

        for item_id, score in zip(items, scores_norm):
            cbf_scores[user][item_id] = score

# Save normalized CBF scores
with open('/content/drive/MyDrive/Movie_Dataset/cbf_scores.pkl', 'wb') as f:
    pickle.dump(cbf_scores, f)

Generating CBF similarity scores...


In [5]:
import pickle
from collections import defaultdict
from scipy.sparse import load_npz

with open('/content/drive/MyDrive/Movie_Dataset/als_scores.pkl', 'rb') as f:
    als_scores = pickle.load(f)

with open('/content/drive/MyDrive/Movie_Dataset/cbf_scores.pkl', 'rb') as f:
    cbf_scores = pickle.load(f)

train_matrix = load_npz('/content/drive/MyDrive/Movie_Dataset/unzipped_data/train_matrix.npz')

alpha = 0.9  # Weight for ALS

hybrid_scores = defaultdict(list)

num_users = train_matrix.shape[0]


for user in range(num_users):
    user_als = als_scores[user]
    user_cbf = cbf_scores[user]
    user_seen = set(train_matrix[user].nonzero()[1])

    # Union of items from both models
    all_items = set(user_als.keys()) | set(user_cbf.keys())

    for item in all_items:
        if item in user_seen:
            continue  # Skip already seen items

        als_score = user_als.get(item, 0)
        cbf_score = user_cbf.get(item, 0)

        hybrid_score = alpha * als_score + (1 - alpha) * cbf_score
        hybrid_scores[user].append((item, hybrid_score))

    # Sort by hybrid score (descending) and keep top 10
    hybrid_scores[user] = sorted(hybrid_scores[user], key=lambda x: x[1], reverse=True)[:10]

    import pickle

with open('/content/drive/MyDrive/Movie_Dataset/hybrid_top10.pkl-2', 'wb') as f:
    pickle.dump(hybrid_scores, f)

print("Saved hybrid recommendations to hybrid_top10.pkl")


Saved hybrid recommendations to hybrid_top10.pkl


In [6]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
import pickle
from sklearn.model_selection import train_test_split
from scipy.sparse import load_npz
from sklearn.preprocessing import LabelEncoder

# Load hybrid scores (if not already in memory)
with open('/content/drive/MyDrive/Movie_Dataset/hybrid_top10.pkl-2', 'rb') as f:
    hybrid_scores = pickle.load(f)

# Load test data
test_df = pd.read_csv('/content/drive/MyDrive/Movie_Dataset/unzipped_data/normalized.csv')
_, test_df = train_test_split(test_df, test_size=0.2, random_state=42)  # same split logic


# Load train matrix
train_matrix = load_npz('/content/drive/MyDrive/Movie_Dataset/unzipped_data/train_matrix.npz')
with open('/content/drive/MyDrive/Movie_Dataset/als_model.pkl', 'rb') as f:
    model = pickle.load(f)

# Build test ground truth (relevant items per user)
user_relevant = defaultdict(set)
for row in test_df.itertuples():
    if row.normalized_rating > 0:  # You may adjust this threshold
        user_relevant[row.userId].add(row.movieId)

# Step 1: Precision, Recall, F1, MAP, NDCG
precisions, recalls, f1s, maps, ndcgs = [], [], [], [], []

for user, recs in hybrid_scores.items():
    rel = user_relevant[user]
    rec_items = [item for item, score in recs]

    if not rel or not recs:
        continue

    hits = [1 if item in rel else 0 for item in rec_items]
    num_hits = sum(hits)

    precision = num_hits / 10
    recall = num_hits / len(rel)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    # MAP@10
    ap = 0.0
    correct = 0
    for i, hit in enumerate(hits, start=1):
        if hit:
            correct += 1
            ap += correct / i
    map_k = ap / min(len(rel), 10)

    # NDCG@10
    dcg = sum([hit / np.log2(idx + 2) for idx, hit in enumerate(hits)])
    ideal_hits = [1] * min(len(rel), 10)
    idcg = sum([1 / np.log2(idx + 2) for idx in range(len(ideal_hits))])
    ndcg = dcg / idcg if idcg != 0 else 0

    # Store
    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)
    maps.append(map_k)
    ndcgs.append(ndcg)

# Step 2: Coverage
all_recommended_items = set()
for recs in hybrid_scores.values():
    for item, _ in recs:
        all_recommended_items.add(item)

num_movies = train_matrix.shape[1]  # Add this line
coverage = len(all_recommended_items) / num_movies

# Step 3: Diversity
# Reload the ratings file to recreate the movie encoder
ratings = pd.read_csv('/content/drive/MyDrive/Movie_Dataset/unzipped_data/normalized.csv')
movie_encoder = LabelEncoder()
movie_encoder.fit(ratings['movieId'])

# Convert original movieIds → encoded indices for ALS
encoded_item_ids = []
for item in all_recommended_items:
    if item in movie_encoder.classes_:
        encoded_index = movie_encoder.transform([item])[0]
        encoded_item_ids.append(encoded_index)

# Compute diversity using ALS item vectors
item_vectors = model.item_factors[encoded_item_ids]
similarity_matrix = cosine_similarity(item_vectors)
diversity = 1 - np.mean(similarity_matrix)


# Step 4: Print results
print("Hybrid Evaluation Results:")
print(f"Precision@10: {np.mean(precisions):.4f}")
print(f"Recall@10:    {np.mean(recalls):.4f}")
print(f"F1@10:        {np.mean(f1s):.4f}")
print(f"MAP@10:       {np.mean(maps):.4f}")
print(f"NDCG@10:      {np.mean(ndcgs):.4f}")
print(f"Coverage:     {coverage:.4f}")
print(f"Diversity:    {diversity:.4f}")


Hybrid Evaluation Results:
Precision@10: 0.2055
Recall@10:    0.1710
F1@10:        0.1446
MAP@10:       0.1788
NDCG@10:      0.2816
Coverage:     0.0614
Diversity:    0.8063
